# Text Embedding using Sentence-BERT (SBERT)
نعمل embedding للـ `clean_text` من الـ dataset باستخدام SBERT

In [1]:
# Install sentence-transformers if not already installed
!pip install sentence-transformers -q

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import time

# Load the cleaned dataset
df = pd.read_csv(r"C:\Users\Abdelrahman\Desktop\embedding\cleaned_sentiment_dataset.csv")
print(f"Dataset shape: {df.shape}")
print(df[['clean_text', 'label']].head())

Dataset shape: (26333, 6)
                                          clean_text                 label
0  anyone else have symptoms much improved after ...               Bipolar
1  i basicaly do not exist sure feels that way no...              Suicidal
2  finding a sense of self its like one minute im...  Personality_disorder
3  putting into words what were just tangles in m...               Anxiety
4  anxiety makes me worry about my public image s...               Anxiety


In [2]:
df = df.dropna(subset=['clean_text'])
df['clean_text'] = df['clean_text'].astype(str)
print(f"Rows after cleaning: {len(df)}")

Rows after cleaning: 26298


In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
# Generate embeddings
print("Generating embeddings...")
start = time.time()

embeddings = model.encode(
    df['clean_text'].tolist(),
    batch_size=64,           
    show_progress_bar=True,  
    convert_to_numpy=True   
)

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s")
print(f"Embeddings shape: {embeddings.shape}")  # (num_samples, 384)

Generating embeddings...


Batches:   0%|          | 0/411 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print("Sample text:", df['clean_text'].iloc[0])
print("Embedding vector (first 10 dims):", embeddings[0][:10])
print("Embedding dimension:", embeddings.shape[1])

In [ ]:
np.save("sbert_embeddings.npy", embeddings)
print("Embeddings saved to sbert_embeddings.npy")

df[['label']].to_csv("embedding_labels.csv", index=False)
print("Labels saved to embedding_labels.csv")

In [ ]:
embedding_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['clean_text', 'label']].reset_index(drop=True), embedding_df], axis=1)
final_df.to_csv("dataset_with_embeddings.csv", index=False)
print(f"Full dataset with embeddings saved! Shape: {final_df.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sample_texts = [
    df['clean_text'].iloc[0],
    df['clean_text'].iloc[1],
    df['clean_text'].iloc[2],
]

sample_embeddings = model.encode(sample_texts)
sim_matrix = cosine_similarity(sample_embeddings)

print("Cosine Similarity Matrix:")
print(np.round(sim_matrix, 3))
for i in range(len(sample_texts)):
    print(f"\n[{i}] {sample_texts[i][:80]}...")

In [ ]:
!pip install umap-learn -q

import umap
import matplotlib.pyplot as plt

sample_size = min(1000, len(embeddings))
sample_idx = np.random.choice(len(embeddings), sample_size, replace=False)
sample_emb = embeddings[sample_idx]
sample_labels = df['label'].iloc[sample_idx].values

reducer = umap.UMAP(n_components=2, random_state=42)
reduced = reducer.fit_transform(sample_emb)

plt.figure(figsize=(10, 7))
unique_labels = np.unique(sample_labels)
for label in unique_labels:
    mask = sample_labels == label
    plt.scatter(reduced[mask, 0], reduced[mask, 1], label=label, alpha=0.6, s=10)

plt.title("SBERT Embeddings - UMAP Visualization")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig("embeddings_umap.png", dpi=150)
plt.show()
print("UMAP visualization saved to embeddings_umap.png")